# 60b — DS5 StM v2: Boltzmann Inversion from V(x) (DS5_02 A2)

**Branch:** DS5_02 Phase A Stage 1, item A2  
**Predecessor:** `41_ds5_statistical_mechanics.ipynb` (original, FLAG A — β=2/σ²)  
**Depends on:** `60_ds5_dys_v2_potential_well.ipynb` (V_curvature_primary → β_v2)

## What changes (A2)

DS5_01 FLAG A (notebook 41, CRITICAL): `β_behav = 2/σ²` — assumes V(x) = x²/2 (parabolic),  
which is only exact for a Gaussian X_t distribution.

v2 replaces this with:
- **β_v2 = V''(x*)** — curvature of the potential well at the stable fixed point x*
- ⚠ There is **no analytic benchmark ratio** between β_v2 and β_behav. β_behav = 2/σ̂² is built from the
  SDE **increment** SD σ̂ (StoP); β_v2 = V''(x*) with V = −log p is built from the **level** dispersion s
  of the X_t distribution (for Gaussian p: V(x) = (x−x*)²/(2s²), so V''(x*) = 1/s², not 1/σ̂²).
  StoP's constant-drift SDE has no stationary distribution, so σ̂ and s are not linked by any fixed constant.
- β_v2 is further attenuated relative to 1/s² because notebook 60 fits the KDE bandwidth on the
  *reflected* sample and then Savitzky-Golay-smooths V: median β_v2 = 2.26 vs median 1/s² = 42.75
  (≈19× attenuation, n = 66).
- For non-Gaussian X_t: V''(x*) reflects the actual local curvature

Downstream quantities:
- `chi_v2 = β_v2 · x* · (1 − x*)` — susceptibility at the attractor (not at the mean)
- `F_v2` = free-energy proxy: V(x*) − (1/β_v2)·log(Z_approx)

## Key review gate question (A2)

DS5_01 finding: condition ordering β_behav: **control > reintervention > default**  
(higher β → more ordered/constrained → less noise).  
Does β_v2 preserve this ordering?

## FLAG A vs v2 relationship

| Method | Formula | Relationship | Non-Gaussian |
|---|---|---|---|
| FLAG A (DS5_01) | β = 2/σ² | correct | parabolic assumption |
| v2 (this nb) | β = V''(x*) | β = 1/s² (level SD s, NOT increment σ̂ — no fixed ratio to β_behav) | actual curvature |

There is no expected factor-of-2 offset — the two β's are built from different variances (increment vs
level) and β_v2 is additionally smoothing-attenuated.  
What matters: (1) relative ordering across participants, (2) condition ordering preservation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

In [ ]:
ROOT    = Path('..')
OUT_DYS = ROOT / 'outputs' / 'model_outputs' / 'dynamical_systems'
OUT_STM = ROOT / 'outputs' / 'model_outputs' / 'statistical_mechanics'
OUT_FIG = ROOT / 'outputs' / 'model_outputs' / 'statistical_mechanics' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load inputs ────────────────────────────────────────────────────────────
# V(x) outputs from notebook 60 (A1)
dys_v2 = pd.read_csv(OUT_DYS / 'participant_dynamics_v2.csv')

# Original StM parameters (FLAG A, β_behav = 2/σ²)
stm_orig = pd.read_csv(OUT_STM / 'participant_stm_parameters.csv')

# Original StoP parameters (for sig_hat reference)
dys_orig = pd.read_csv(OUT_DYS / 'participant_dynamics_parameters.csv',
                       usecols=['participant_id_canonical', 'study_condition',
                                'sig_hat', 'x_mean'])

print(f'DyS v2 records: {len(dys_v2)}')
print(f'StM orig records: {len(stm_orig)}')
print(f'\nV_curvature_primary NaN: {dys_v2["V_curvature_primary"].isna().sum()}')
print(f'Has stable FP: {dys_v2["has_stable_fp"].sum()}')

In [ ]:
# ── Compute β_v2 and χ_v2 ─────────────────────────────────────────────────
# β_v2 = V''(x*) at stable FP (curvature of potential well)
# Only defined for participants with a stable fixed point (has_stable_fp=True)
#
# χ_v2 = β_v2 · x* · (1 − x*)   (susceptibility at attractor state)
# x* = x_fp_primary (location of primary stable FP)

stm_v2 = dys_v2[['participant_id_canonical', 'study_condition',
                  'has_stable_fp', 'low_confidence',
                  'x_fp_primary', 'V_curvature_primary', 'p_at_fp_primary']].copy()

# β_v2 = V_curvature_primary (only meaningful when positive = stable FP)
stm_v2['beta_v2'] = np.where(
    stm_v2['has_stable_fp'] & stm_v2['V_curvature_primary'].notna(),
    stm_v2['V_curvature_primary'],
    np.nan
)

# χ_v2 = β_v2 · x* · (1 − x*)
stm_v2['chi_v2'] = stm_v2['beta_v2'] * stm_v2['x_fp_primary'] * (1 - stm_v2['x_fp_primary'])

# Flag: eligible for v2 StM (has stable FP and not low confidence)
stm_v2['is_stm_eligible_v2'] = stm_v2['has_stable_fp'] & ~stm_v2['low_confidence']

print(f'β_v2 defined for: {stm_v2["beta_v2"].notna().sum()} participants')
print(f'χ_v2 defined for: {stm_v2["chi_v2"].notna().sum()} participants')
print(f'High-confidence v2 StM eligible: {stm_v2["is_stm_eligible_v2"].sum()}')
print(f'\nβ_v2 summary (eligible):')
print(stm_v2.loc[stm_v2['beta_v2'].notna(), 'beta_v2'].describe().round(3))

In [ ]:
# ── Merge with original StM for comparison ────────────────────────────────
compare = (
    stm_v2
    .merge(stm_orig[['participant_id_canonical', 'beta_behav', 'chi_behav', 'F_behav']],
           on='participant_id_canonical', how='left')
    .merge(dys_orig[['participant_id_canonical', 'sig_hat', 'x_mean']],
           on='participant_id_canonical', how='left')
)

# ⚠ No expected ratio: β_behav = 2/σ̂_increment², β_v2 = V''(x*) ≈ 1/s_level² for Gaussian.
# Different variances; plus β_v2 is attenuated by nb60's reflected-sample bandwidth and
# Savitzky-Golay smoothing. The ratio below is descriptive only.
compare['beta_ratio'] = compare['beta_behav'] / compare['beta_v2']

elig = compare[compare['beta_v2'].notna()].copy()
print(f'Participants with both β_behav and β_v2: {len(elig)}')
print(f'\nβ ratio (β_behav / β_v2) — descriptive only, no analytic benchmark applies:')
print(elig['beta_ratio'].describe().round(3))
print(f'\nSpearman ρ(β_behav, β_v2): {spearmanr(elig["beta_behav"].dropna(), elig.loc[elig["beta_behav"].notna(), "beta_v2"]).statistic:.3f}')

In [ ]:
# ── Condition ordering: β_v2 by study condition ───────────────────────────
# DS5_01 finding: β_behav control > reintervention > default
# Does β_v2 preserve this ordering?

print('=== REVIEW GATE: CONDITION ORDERING ===')
print()
print('β_behav (FLAG A, original) by condition:')
orig_order = stm_orig.groupby('study_condition')['beta_behav'].median()
print(orig_order.sort_values(ascending=False).round(2))

print()
print('β_v2 (KDE curvature) by condition (only participants with stable FP):')
v2_order = compare.dropna(subset=['beta_v2']).groupby('study_condition')['beta_v2'].median()
print(v2_order.sort_values(ascending=False).round(3))

print()
print('N per condition with β_v2 defined:')
print(compare.dropna(subset=['beta_v2']).groupby('study_condition')['beta_v2'].count())

from scipy.stats import kruskal
_e = compare.dropna(subset=['beta_v2'])
_groups = [_e.loc[_e['study_condition'] == c, 'beta_v2'].values
           for c in ['control', 'default', 'reintervention']]
_H, _p = kruskal(*_groups)
print()
print(f'Kruskal-Wallis on β_v2 by condition: H = {_H:.3f}, p = {_p:.4f}')
print('⚠ NOT significant — the median ordering is directionally consistent but is NOT')
print('  statistically distinguishable from chance at this N (16 / 25 / 25).')
print()
print('Like-for-like check — β_behav medians restricted to the SAME 66 participants:')
print(_e.groupby('study_condition')['beta_behav'].median().sort_values(ascending=False).round(2))

In [ ]:
# ── χ_v2 vs χ_behav comparison ────────────────────────────────────────────
print('χ_behav (original) by condition (median):')
print(stm_orig.groupby('study_condition')['chi_behav'].median().round(3))

print()
print('χ_v2 (KDE-based) by condition (median):')
print(compare.dropna(subset=['chi_v2']).groupby('study_condition')['chi_v2'].median().round(4))

both_chi = compare.dropna(subset=['chi_v2', 'chi_behav'])
if len(both_chi) > 5:
    rho, p = spearmanr(both_chi['chi_behav'], both_chi['chi_v2'])
    print(f'\nSpearman ρ(χ_behav, χ_v2): {rho:.3f} (p={p:.3f}, n={len(both_chi)})')

In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
COND_COLORS = {'control': '#4dac26', 'default': '#d01c8b', 'reintervention': '#f1b6da'}

# Plot 1: β_behav vs β_v2 scatter
ax = axes[0]
elig_plot = compare.dropna(subset=['beta_v2', 'beta_behav'])
for cond, color in COND_COLORS.items():
    sub = elig_plot[elig_plot['study_condition'] == cond]
    ax.scatter(sub['beta_behav'], sub['beta_v2'], color=color,
               alpha=0.5, s=25, label=cond)
# No Gaussian reference line: β_behav and β_v2 are built from different variances
# (SDE increment vs. X_t level), so no analytic benchmark ratio exists.
ax.set_xlabel('β_behav (FLAG A: 2/σ²)')
ax.set_ylabel('β_v2 (KDE: V\'\'(x*))')
ax.set_title('β comparison')
ax.legend(fontsize=7)

# Plot 2: β_v2 by condition boxplot
ax = axes[1]
cond_order = ['control', 'reintervention', 'default']
data_boxes = [compare.loc[compare['study_condition']==c, 'beta_v2'].dropna().values
              for c in cond_order]
bp = ax.boxplot(data_boxes, patch_artist=True, notch=False)
for patch, cond in zip(bp['boxes'], cond_order):
    patch.set_facecolor(COND_COLORS[cond])
    patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(cond_order, fontsize=9)
ax.set_ylabel('β_v2')
ax.set_title('β_v2 by condition')
ax.text(0.5, 0.92, 'DS5_01 ordering: control > reintervention > default',
        transform=ax.transAxes, ha='center', fontsize=7.5, color='#555')

# Plot 3: χ_v2 by condition boxplot
ax = axes[2]
data_chi = [compare.loc[compare['study_condition']==c, 'chi_v2'].dropna().values
            for c in cond_order]
bp2 = ax.boxplot(data_chi, patch_artist=True)
for patch, cond in zip(bp2['boxes'], cond_order):
    patch.set_facecolor(COND_COLORS[cond])
    patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(cond_order, fontsize=9)
ax.set_ylabel('χ_v2')
ax.set_title('χ_v2 by condition (susceptibility at attractor)')

fig.suptitle('DS5 StM v2 — Boltzmann Inversion from V(x) (A2)', fontsize=12)
plt.tight_layout()
fig.savefig(OUT_FIG / 'fig_stm_v2_beta_comparison.png', bbox_inches='tight', dpi=200)
print('Saved: fig_stm_v2_beta_comparison.png')
plt.show()

In [ ]:
# ── Save v2 StM parameters ─────────────────────────────────────────────────
out_cols = [
    'participant_id_canonical', 'study_condition',
    'has_stable_fp', 'is_stm_eligible_v2', 'low_confidence',
    'x_fp_primary', 'beta_v2', 'chi_v2',
]
out_path = OUT_STM / 'participant_stm_parameters_v2.csv'
stm_v2[out_cols].to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Rows: {len(stm_v2)}')

In [ ]:
# ── A1+A2 combined review gate verdict ────────────────────────────────────
print('=' * 65)
print('DS5_02 PHASE A STAGE 1 — A1+A2 REVIEW GATE VERDICT')
print('=' * 65)
print()

n_orig_det = 154   # from DS5_01 FLAG2 (median split)
n_total    = 414
n_v2_stable = int(stm_v2['has_stable_fp'].sum())

print('A1 — DyS: Fixed-point classification')
print(f'  Original FLAG2 (median split): {n_orig_det}/{n_total} deterministic')
print(f'  v2 KDE stable FP:              {n_v2_stable}/{n_total}')
from sklearn.metrics import cohen_kappa_score
from scipy.stats import fisher_exact
_agree_df = stm_v2[['participant_id_canonical', 'has_stable_fp']].merge(
    stm_orig[['participant_id_canonical', 'behaviorally_deterministic']],
    on='participant_id_canonical')
_a = _agree_df['behaviorally_deterministic'].astype(bool)
_b = _agree_df['has_stable_fp'].astype(bool)
_ct = pd.crosstab(_a, _b)
_raw = float((_a == _b).mean())
_kappa = cohen_kappa_score(_a, _b)
_odds, _pfish = fisher_exact(_ct.values)
print(f'  Raw agreement with original:   {_raw:.1%}  (chance level for these marginals ~59%)')
print(f'  Cohen kappa:                   {_kappa:+.3f}   Fisher exact p = {_pfish:.3f}')
print(f'  Crosstab (rows=behaviorally_deterministic, cols=has_stable_fp):')
print(_ct.to_string())
print(f'  FINDING: the two labels are STATISTICALLY UNRELATED (kappa ~ 0, p = {_pfish:.2f}).')
print(f'           v2 is NOT a refinement of the DS5_01 set — it identifies a different construct.')
print(f'           Only 23 of the 154 original "deterministic" have a stable FP, while 43 of')
print(f'           the 260 "non-deterministic" do.')
print()

orig_order_str = ' > '.join(
    stm_orig.groupby('study_condition')['beta_behav'].median()
    .sort_values(ascending=False).index.tolist()
)
v2_meds = compare.dropna(subset=['beta_v2']).groupby('study_condition')['beta_v2'].median()
v2_order_str   = ' > '.join(v2_meds.sort_values(ascending=False).index.tolist())

print('A2 — StM: β ordering by condition')
print(f'  Original β_behav ordering: {orig_order_str}')
print(f'  v2 β_v2 ordering:          {v2_order_str}')
order_preserved = (orig_order_str == v2_order_str)
print(f'  Ordering direction matches: {order_preserved}')
print(f'  ⚠ NOT a statistical result: Kruskal-Wallis on β_v2 by condition H = 4.346, p = 0.114.')
print(f'    Honest verdict: directionally consistent, not statistically supported at N = 16/25/25.')
print(f'    (β_behav ordering above is over all 414; restricted to the same 66 it is still')
print(f'     control 155.62 > reintervention 89.22 > default 48.79.)')

n_v2_elig = int(stm_v2['beta_v2'].notna().sum())
print(f'  Caveat: β_v2 defined only for {n_v2_elig}/{n_total} participants.')
print(f'          Small N per condition may make ordering fragile.')
print()
print('OVERALL A1+A2:')
print('  MATERIAL CHANGE in DyS classification (66 vs 154 → 58% agreement).')
print('  β_v2 condition ordering: check plot and value above.')
print('  Key caveat: only 66/414 have stable FP → v2 StM covers far fewer participants.')
print('  Interpretation: most DS5 participants do NOT show clear attractor dynamics')
print('  in their 30-45 day X_t series — consistent with noise-dominated SDE finding.')

## Summary

**Outputs:**
- `outputs/model_outputs/statistical_mechanics/participant_stm_parameters_v2.csv`
- `outputs/model_outputs/statistical_mechanics/figures/fig_stm_v2_beta_comparison.png`

**A2 result:** β_v2 = V''(x*) replaces FLAG A β=2/σ².  
- No benchmark ratio applies: β_behav uses the SDE increment variance, β_v2 the X_t level curvature
  (and β_v2 is smoothing-attenuated). Observed β_behav/β_v2: median 34.4, min 2.5, max 691.8.  
- Condition ordering: see review gate cell above

**Combined A1+A2 finding:**  
Only 66/414 participants show a stable KDE attractor.  
The majority are noise-dominated (no clear V(x) minimum) — consistent with DS5_01's  
StoP finding that σ̂ > μ̂ for most participants (σ/μ ratio > 1).  

**For the review gate:** This is a material clarification, and a stronger one than 'refinement'.  
The v2 stable-FP set (66/414) and DS5_01's median-split 'deterministic' set (154/414) are
statistically unrelated (raw agreement 58.0% — at chance for these marginals; Cohen's kappa
= −0.018; Fisher exact p = 0.781; crosstab [[217, 43], [131, 23]]). Only 23 of the 154
original 'deterministic' participants have a stable V(x) attractor. v2 therefore identifies a
**different construct**, not a conservative subset of the same one — any downstream v2 result
(nb63, nb64) must be interpreted on that basis.  

**Next:** Decide whether to proceed to Phase B (re-cluster with v2 features) or document  
A1+A2+A3 as the Stage 1 findings and close the DS5_02 Phase A Stage 1 gate.